# Workshop notebook 2 — A minimal ODE-constrained mixture model

This notebook fits a small, ready-to-run **ODE-constrained mixture model (ODE-MM)** to simulated snapshot data.

## Model idea

We assume two hidden subpopulations.  
Each subpopulation has a mean trajectory

$\mu_i(t) = B_\infty - (B_\infty - B_0)e^{-k_i t}$

and the observed snapshot data are drawn from a Gaussian mixture

$ p(y \mid t) = w\,\mathcal N(y; \mu_1(t), \sigma) + (1-w)\,\mathcal N(y; \mu_2(t), \sigma) $

This is not the full paper implementation, but it captures the core structure:
- **mixture weights**
- **subpopulation-specific dynamics**
- **joint fit across all time points**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

rng = np.random.default_rng(7)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## 1. Simulate a workshop dataset
Generate a dataset for 120 cells to ensure that MM fails

In [ ]:
times = np.array([0.0, 0.1, 0.2, 0.3, 0.5, 1.0])

def mean_response(t, k, B0=0.22, B_inf=0.48):
    return B_inf - (B_inf - B0) * np.exp(-k * t)

truth = {
    "w": 0.70,      # fraction of subpopulation 1
    "k1": 0.45,     # slower response
    "k2": 2.15,     # faster response
    "sigma": 0.022
}

def generate_dataset(n_cells=400):
    data = {}
    for t in times:
        z = rng.binomial(1, 1-truth["w"], size=n_cells)  # z=1 means subpopulation 2
        mu1 = mean_response(t, truth["k1"])
        mu2 = mean_response(t, truth["k2"])
        mu = np.where(z == 0, mu1, mu2)
        y = rng.normal(mu, truth["sigma"], size=n_cells)
        data[t] = y
    return data

data = generate_dataset(n_cells=120)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
axes = axes.ravel()
for ax, t in zip(axes, times):
    ax.hist(data[t], bins=35, density=True, alpha=0.55)
    ax.set_title(f"t = {t:g}")
    ax.set_xlabel("measured B")
    ax.set_ylabel("density")
fig.suptitle("Simulated workshop dataset", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

## 2. Define the joint negative log-likelihood

The same parameter vector explains **all time points simultaneously**.

In [ ]:
def unpack(theta):
    # unconstrained -> constrained parameters
    a_w, a_k1, a_k2, a_sigma = theta
    w = 1 / (1 + np.exp(-a_w))            # (0, 1)
    k1 = np.exp(a_k1)                     # > 0
    k2 = np.exp(a_k2)                     # > 0
    sigma = np.exp(a_sigma)               # > 0
    return w, k1, k2, sigma

def neg_loglik(theta, data, times):
    w, k1, k2, sigma = unpack(theta)
    eps = 1e-12
    nll = 0.0
    for t in times:
        y = data[t]
        mu1 = mean_response(t, k1)
        mu2 = mean_response(t, k2)
        dens = w * norm.pdf(y, mu1, sigma) + (1 - w) * norm.pdf(y, mu2, sigma)
        nll -= np.sum(np.log(dens + eps))
    return nll

## 3. Fit the ODE-MM with multistart optimization

The paper uses multi-start local optimization for the ODE-constrained likelihood problem.
Here we do the same in a small teaching example.

In [ ]:
def fit_multistart(data, times, n_starts=20, seed=1):
    rng = np.random.default_rng(seed)
    best = None
    history = []

    print(n_starts)

    for _ in range(n_starts):
        theta0 = np.array([
            rng.normal(0, 1.0),           # logit(w)
            rng.normal(np.log(0.6), 0.8), # log(k1)
            rng.normal(np.log(1.5), 0.8), # log(k2)
            rng.normal(np.log(0.03), 0.5) # log(sigma)
        ])
        res = minimize(neg_loglik, theta0, args=(data, times), method="L-BFGS-B")
        history.append(res.fun)
        if best is None or res.fun < best.fun:
            best = res

    return best, np.array(history)

best, history = fit_multistart(data, times, n_starts=30, seed=2)
best.fun
best

In [ ]:
w_hat, k1_hat, k2_hat, sigma_hat = unpack(best.x)

# reorder for interpretation: subpopulation 1 = slower one
if k1_hat > k2_hat:
    k1_hat, k2_hat = k2_hat, k1_hat
    w_hat = 1 - w_hat

print("Estimated parameters")
print(f"w      = {w_hat:.3f}   (truth {truth['w']:.3f})")
print(f"k1     = {k1_hat:.3f}   (truth {truth['k1']:.3f})")
print(f"k2     = {k2_hat:.3f}   (truth {truth['k2']:.3f})")
print(f"sigma  = {sigma_hat:.3f}   (truth {truth['sigma']:.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(history, bins=12, alpha=0.7)
ax.axvline(best.fun, lw=3, label="best objective")
ax.set_xlabel("negative log-likelihood")
ax.set_ylabel("count")
ax.set_title("Multistart optimization results")
ax.legend()
plt.show()

## 4. Visualize the fitted subpopulation trajectories

Unlike the independent GMM approach, the inferred means are now linked through the same mechanistic form across all time points.

In [ ]:
true_mu1 = np.array([mean_response(t, truth["k1"]) for t in times])
true_mu2 = np.array([mean_response(t, truth["k2"]) for t in times])
fit_mu1  = np.array([mean_response(t, k1_hat) for t in times])
fit_mu2  = np.array([mean_response(t, k2_hat) for t in times])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(times, true_mu1, "o--", lw=2, label="true slow")
ax.plot(times, true_mu2, "o--", lw=2, label="true fast")
ax.plot(times, fit_mu1, "s-", lw=1, label="fit slow")
ax.plot(times, fit_mu2, "s-", lw=1, label="fit fast")
ax.set_xlabel("time")
ax.set_ylabel("subpopulation mean")
ax.set_title("ODE-constrained mean trajectories")
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
axes = axes.ravel()
xs = np.linspace(0.15, 0.55, 500)

for ax, t in zip(axes, times):
    y = data[t]
    mu1 = mean_response(t, k1_hat)
    mu2 = mean_response(t, k2_hat)

    comp1 = w_hat * norm.pdf(xs, mu1, sigma_hat)
    comp2 = (1 - w_hat) * norm.pdf(xs, mu2, sigma_hat)
    total = comp1 + comp2

    ax.hist(y, bins=35, density=True, alpha=0.4, color="0.75", label="data")
    ax.plot(xs, comp1, lw=2, label="subpop 1" if t == times[0] else None)
    ax.plot(xs, comp2, lw=2, label="subpop 2" if t == times[0] else None)
    ax.plot(xs, total, "k--", lw=2, label="mixture" if t == times[0] else None)
    ax.set_title(f"t = {t:g}")
    ax.set_xlabel("measured B")
    ax.set_ylabel("density")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.05))
fig.suptitle("Joint ODE-constrained mixture fit", y=1.09, fontsize=14)
fig.tight_layout()
plt.show()

## 5. Mini tasks for participants

Change one thing at a time and rerun the fit.

### Task A — harder separation
Set
```python
truth["k2"] = 1.1
```
What happens to the fit?

### Task B — more noise
Set
```python
truth["sigma"] = 0.04
```
Can the model still recover two subpopulations?

### Task C — smaller sample size
Call
```python
data = generate_dataset(n_cells=120, rng=rng)
```
How stable are the estimates across repeated runs?

### Task D — wrong model (optional or home work)
Modify the likelihood to force a single population.  
Compare the fit visually. Where does it fail?

## 6. Take-home message

This notebook shows the core logic of ODE-MM:

- mixture components represent hidden subpopulations
- subpopulation means are not free at each time point
- they are linked by a mechanistic dynamic model
- fitting all snapshots jointly improves interpretability

This is the main conceptual contribution of the paper.